In [145]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import json
import tiktoken

REPO_DIR = os.path.join("/Users/haya1/Documents/LanguageModel_Labels/congressional_bills/")
# REPO_DIR = "."
os.chdir(REPO_DIR)

load_dotenv(os.path.join(REPO_DIR, ".env"), override=True)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
PER_BATCH_LIMIT = 50e3

In [146]:
def create_embed_requests(descriptions, custom_ids, out_dir, embedding_model="text-embedding-3-small", prefix=""):
    batches = []
    requests = []
    part = 0

    I = len(descriptions)
    for i in range(I):
        request = {
                "custom_id": str(custom_ids[i]),
                "method": "POST",
                "url": "/v1/embeddings",
                "body": {
                    "input": descriptions[i],
                    "model": embedding_model
                }
            }
        requests.append(request)

        if ((len(requests)==PER_BATCH_LIMIT) | (i==(I-1))):
            part = part + 1
            part_path = os.path.join(out_dir, f"{prefix}part{part}.jsonl")

            with open(part_path, "w") as f:
                for request in requests:
                    f.write(json.dumps(request) + "\n")
                print(f"Saved {os.path.basename(part_path)}, n = {len(requests)}, at {os.path.dirname(part_path)}")
            requests = []

            batch = {
                'file': part_path,
                'part': part
            }
            batches.append(batch)

    return(pd.json_normalize(batches))

In [147]:
def query_embeddings(batches):
    client = OpenAI(api_key=OPENAI_API_KEY)

    batches_id = []
    for _, batch in batches.iterrows():
        batch_input_file = client.files.create(
            file = open(batch['file'], "rb"),
            purpose = "batch"
        )

        new_batch = client.batches.create(
            input_file_id = batch_input_file.id,
            endpoint = "/v1/embeddings",
            completion_window = "24h",
            metadata = {"description": f"{os.path.basename(batch['file'])}"}
        )

        batches_id.append(new_batch.id)

    batches['id'] = batches_id
    return(batches)

In [148]:
# Check status of all batches
def check_batches_status(batches):
    client = OpenAI(api_key=OPENAI_API_KEY)
    for _, batch in batches.iterrows():
        file = batch['file']
        id = batch['id']
        batch_status = client.batches.retrieve(id)
        print(f"{os.path.basename(file):>52s}: {batch_status.status}")


In [149]:
def download_batched_responses(batches, out_dir):
    client = OpenAI(api_key=OPENAI_API_KEY)
    for _, batch in batches.iterrows():
        batch_id = batch['id']
        part = batch['part']
        responses_batched_path = os.path.join(out_dir, f"part{part}.jsonl")

        batch_status = client.batches.retrieve(batch_id)
        if batch_status.status != "completed":
            print(f"Skipping incomplete file = {os.path.basename(responses_batched_path)}, Batch ID = {batch_id}")
            continue
        
        output_file_id = batch_status.output_file_id
        responses_batched = client.files.content(output_file_id)
        responses_batched.write_to_file(responses_batched_path)
        print(f"Saved {os.path.basename(responses_batched_path)} at {os.path.dirname(responses_batched_path)}")

In [150]:
def merge_batched_responses(responses_batched_paths):
    responses = []
    for responses_batched_path in sorted(responses_batched_paths):
        responses_batched_file = pd.read_json(responses_batched_path, lines=True)
        print(f"Loaded {os.path.basename(responses_batched_path)}, n = {len(responses_batched_file)}")
        responses.append(responses_batched_file)
    responses = pd.concat(responses)
    return(responses)

In [151]:
def decode_embed_responses(responses):
    responses_decoded = []
    for _, response in responses.iterrows(): 
        response_out =  response.response['body']['data'][0]
        responses_decoded.append({
            "custom_id": response["custom_id"],
            "Embeddings": response_out['embedding'],
            "Tokens": int(response.response["body"]["usage"]["prompt_tokens"])
        })
    return(pd.json_normalize(responses_decoded))


In [152]:
def count_tokens(text, model="text-embedding-3-small"):
    encoding = tiktoken.encoding_for_model(model)
    n_tokens = len(encoding.encode(text)) 
    return(n_tokens)

def estimate_cost(descriptions, model="text-embedding-3-small", batched=True):
    n_tokens = np.zeros_like(descriptions)
    for i in range(len(descriptions)):
        n_tokens[i] = count_tokens(descriptions[i], model)
    
    # Cost without using Batch API
    embed_token_cost = {
        'text-embedding-3-small': 0.020/1e6,
        'text-embedding-3-large': 0.130/1e6,
        'ada v2': 0.100/1e6
    }

    total_cost  = (n_tokens * embed_token_cost[model]).sum()
    if batched:
        total_cost = total_cost/2
    return total_cost

In [153]:
data_dir = os.path.join(REPO_DIR, "Data/Prediction")
temp_dir = os.path.join(REPO_DIR, "Temp/Prediction")

bills_llm_completion = pd.read_csv(os.path.join(data_dir, "bills_llm_completion.csv"))

In [154]:
cost_description = estimate_cost(bills_llm_completion["DescriptionClean"], batched=True)
print(f"Estimated cost to embed Description using Batch API is ${cost_description:.2f}")

cost_description_llm = estimate_cost(bills_llm_completion["DescriptionLLMClean"], batched=True)
print(f"Estimated cost to embed DescriptionLLM using Batch API is ${cost_description_llm:.2f}")

Estimated cost to embed Description using Batch API is $0.00
Estimated cost to embed DescriptionLLM using Batch API is $0.01


# Description

In [155]:
requests_dir = os.path.join(temp_dir, "DescriptionClean/Requests")
os.makedirs(requests_dir, exist_ok=True)

batches = create_embed_requests(bills_llm_completion["DescriptionClean"], bills_llm_completion["ID"], requests_dir, embedding_model="text-embedding-3-small", prefix="description_")
batches_path = os.path.join(temp_dir, "DescriptionClean/batches.csv")
batches.to_csv(batches_path, index=False)
print(f"Created batched prompts with batch details stored at {os.path.basename(batches_path)}, n = {len(batches)}, at {os.path.dirname(batches_path)}")

Saved description_part1.jsonl, n = 39999, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/DescriptionClean/Requests
Created batched prompts with batch details stored at batches.csv, n = 1, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/DescriptionClean


In [156]:
batches = query_embeddings(batches)
batches.to_csv(batches_path, index=False)
print(f"Added batch_id to {os.path.basename(batches_path)}, n = {len(batches)}, at {os.path.dirname(batches_path)}")

Added batch_id to batches.csv, n = 1, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/DescriptionClean


In [173]:
batches = pd.read_csv(os.path.join(temp_dir, "DescriptionClean/batches.csv"))
check_batches_status(batches)

                             description_part1.jsonl: completed


In [174]:
batches = pd.read_csv(os.path.join(temp_dir, "DescriptionClean/batches.csv"))
responses_dir = os.path.join(temp_dir, "DescriptionClean/Responses")
os.makedirs(responses_dir, exist_ok=True)
download_batched_responses(batches, responses_dir)

responses_batched_paths = glob.glob(os.path.join(responses_dir, '*.jsonl'))
responses = merge_batched_responses(responses_batched_paths)

responses_decoded = decode_embed_responses(responses)
responses_decoded.rename(columns={
    'custom_id': 'ID',
    'Embeddings': 'DescriptionEmbed',
    'Tokens': 'DescriptionTokens'
    }, inplace=True)
embedded_descriptions_path = os.path.join(temp_dir, "DescriptionClean/embedded_descriptions.csv")
responses_decoded.to_csv(embedded_descriptions_path, index=False)
print(f"Saved {os.path.basename(embedded_descriptions_path)}, n = {len(responses_decoded)}, at {os.path.dirname(embedded_descriptions_path)}")

Saved part1.jsonl at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/DescriptionClean/Responses
Loaded part1.jsonl, n = 39999
Saved embedded_descriptions.csv, n = 39999, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/DescriptionClean


# DescriptionLLM

In [157]:
requests_dir = os.path.join(temp_dir, "DescriptionLLMClean/Requests")
os.makedirs(requests_dir, exist_ok=True)

batches = create_embed_requests(bills_llm_completion["DescriptionLLMClean"], bills_llm_completion["ID"], requests_dir, embedding_model="text-embedding-3-small", prefix="description_llm_")
batches_path = os.path.join(temp_dir, "DescriptionLLMClean/batches.csv")
batches.to_csv(batches_path, index=False)
print(f"Created batched prompts with batch details stored at {os.path.basename(batches_path)}, n = {len(batches)}, at {os.path.dirname(batches_path)}")

Saved description_llm_part1.jsonl, n = 39999, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/DescriptionLLMClean/Requests
Created batched prompts with batch details stored at batches.csv, n = 1, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/DescriptionLLMClean


In [158]:
batches = query_embeddings(batches)
batches.to_csv(batches_path, index=False)
print(f"Added batch_id to {os.path.basename(batches_path)}, n = {len(batches)}, at {os.path.dirname(batches_path)}")

Added batch_id to batches.csv, n = 1, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/DescriptionLLMClean


In [171]:
batches = pd.read_csv(os.path.join(temp_dir, "DescriptionLLMClean/batches.csv"))
check_batches_status(batches)

                         description_llm_part1.jsonl: completed


In [172]:
batches = pd.read_csv(os.path.join(temp_dir, "DescriptionLLMClean/batches.csv"))

responses_dir = os.path.join(temp_dir, "Responses")
os.makedirs(responses_dir, exist_ok=True)
download_batched_responses(batches, responses_dir)

responses_batched_paths = glob.glob(os.path.join(responses_dir, '*.jsonl'))
responses = merge_batched_responses(responses_batched_paths)

responses_decoded = decode_embed_responses(responses)
responses_decoded.rename(columns={
    'custom_id': 'ID',
    'Embeddings': 'DescriptionLLMEmbed',
    'Tokens': 'DescriptionLLMTokens'
    }, inplace=True)
embedded_descriptions_path = os.path.join(temp_dir, "DescriptionLLMClean/embedded_descriptions_llm.csv")
responses_decoded.to_csv(embedded_descriptions_path, index=False)
print(f"Saved {os.path.basename(embedded_descriptions_path)}, n = {len(responses_decoded)}, at {os.path.dirname(embedded_descriptions_path)}")

Saved part1.jsonl at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/Responses
Loaded part1.jsonl, n = 39999
Saved embedded_descriptions_llm.csv, n = 39999, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Temp/Prediction/DescriptionLLMClean
